# U-Net Architecture for Brain Tumor Segmentation (BraTS)
Supports 2D and 3D modes.
Output classes: 3 (background, tumor core, whole tumor)
               or 4 (BraTS: background, NCR/NET, ED, ET)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

# 1. BUILDING BLOCKS

In [3]:
class DoubleConv(nn.Module):
    """Two consecutive Conv → BN → ReLU blocks."""
    def __init__(self, in_ch, out_ch, mid_ch=None):
        super().__init__()
        mid_ch = mid_ch or out_ch
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
 
    def forward(self, x):
        return self.block(x)
 
 
class ResDoubleConv(nn.Module):
    """Double conv with residual shortcut (optional but recommended for tumors)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.skip = (
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
            if in_ch != out_ch else nn.Identity()
        )
        self.bn = nn.BatchNorm2d(out_ch) if in_ch != out_ch else nn.Identity()
 
    def forward(self, x):
        return F.relu(self.conv(x) + self.bn(self.skip(x)), inplace=True)
 
 
class AttentionGate(nn.Module):
    """
    Soft attention gate on skip connections.
    Suppresses irrelevant background features before concatenation.
    """
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.Wg = nn.Sequential(nn.Conv2d(g_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch))
        self.Wx = nn.Sequential(nn.Conv2d(x_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch))
        self.psi = nn.Sequential(nn.Conv2d(inter_ch, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
 
    def forward(self, g, x):
        # g = gating signal (from decoder), x = skip feature
        g_up = F.interpolate(self.Wg(g), size=x.shape[2:], mode="bilinear", align_corners=False)
        alpha = self.psi(F.relu(g_up + self.Wx(x), inplace=True))
        return x * alpha
 
 
class Down(nn.Module):
    """MaxPool → DoubleConv (encoder step)."""
    def __init__(self, in_ch, out_ch, residual=True):
        super().__init__()
        conv = ResDoubleConv if residual else DoubleConv
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), conv(in_ch, out_ch))
 
    def forward(self, x):
        return self.pool_conv(x)
 
 
class Up(nn.Module):
    """
    Upsample → concat skip → DoubleConv (decoder step).
    Supports bilinear upsampling or transposed convolution.
    """
    def __init__(self, in_ch, out_ch, bilinear=True, residual=True, attention=True):
        super().__init__()
        self.attention = attention
        conv = ResDoubleConv if residual else DoubleConv
 
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_ch, out_ch, in_ch // 2)
        else:
            self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
            self.conv = conv(in_ch, out_ch)
 
        if attention:
            self.att = AttentionGate(in_ch // 2, in_ch // 2, in_ch // 4)
 
    def forward(self, x, skip):
        x = self.up(x)
        if self.attention:
            skip = self.att(x, skip)
        # Pad if spatial dims don't match exactly
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.cat([skip, x], dim=1))
 
 
class OutConv(nn.Module):
    def __init__(self, in_ch, n_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, n_classes, kernel_size=1)
 
    def forward(self, x):
        return self.conv(x)

# 2. FULL U-NET MODEL

In [5]:
class UNet(nn.Module):
    """
    U-Net for brain tumor segmentation.
 
    Args:
        in_channels  : number of MRI modalities (4 for BraTS: T1, T1ce, T2, FLAIR)
        n_classes    : 4 for BraTS sub-regions (bg, NCR/NET, edema, enhancing)
        base_filters : feature maps in first encoder block (default 64)
        bilinear     : True = bilinear upsample, False = transposed conv
        residual     : use residual double-conv blocks
        attention    : use attention gates on skip connections
        deep_supervision: return auxiliary outputs from deeper decoder levels
    """
    def __init__(
        self,
        in_channels: int = 4,
        n_classes: int = 4,
        base_filters: int = 64,
        bilinear: bool = True,
        residual: bool = True,
        attention: bool = True,
        deep_supervision: bool = False,
    ):
        super().__init__()
        self.deep_supervision = deep_supervision
        f = base_filters
        conv = ResDoubleConv if residual else DoubleConv
 
        # ── Encoder ──
        self.inc   = conv(in_channels, f)          # 256×256 → 256×256, 64ch
        self.down1 = Down(f,     f * 2,  residual) # 128×128, 128ch
        self.down2 = Down(f * 2, f * 4,  residual) #  64×64,  256ch
        self.down3 = Down(f * 4, f * 8,  residual) #  32×32,  512ch
 
        # ── Bottleneck ──
        factor = 2 if bilinear else 1
        self.down4 = Down(f * 8, f * 16 // factor, residual)  # 16×16, 1024ch
 
        # ── Decoder ──
        self.up1 = Up(f * 16, f * 8  // factor, bilinear, residual, attention)
        self.up2 = Up(f * 8,  f * 4  // factor, bilinear, residual, attention)
        self.up3 = Up(f * 4,  f * 2  // factor, bilinear, residual, attention)
        self.up4 = Up(f * 2,  f,                bilinear, residual, attention)
 
        # ── Output heads ──
        self.outc = OutConv(f, n_classes)
 
        if deep_supervision:
            self.ds2 = OutConv(f,               n_classes)
            self.ds3 = OutConv(f * 4 // factor, n_classes)
 
    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
 
        # Decoder
        d4 = self.up1(x5, x4)
        d3 = self.up2(d4, x3)
        d2 = self.up3(d3, x2)
        d1 = self.up4(d2, x1)
 
        logits = self.outc(d1)
 
        if self.deep_supervision and self.training:
            ds2 = F.interpolate(self.ds2(d2), size=logits.shape[2:], mode="bilinear", align_corners=False)
            ds3 = F.interpolate(self.ds3(d3), size=logits.shape[2:], mode="bilinear", align_corners=False)
            return logits, ds2, ds3  # main + two auxiliary outputs
 
        return logits

# 3. LOSS FUNCTIONS

In [6]:
class DiceLoss(nn.Module):
    """
    Soft Dice loss per class, averaged across classes.
    Works well with class imbalance common in tumor segmentation.
    """
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth
 
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        n_classes = probs.shape[1]
        targets_oh = F.one_hot(targets, n_classes).permute(0, 4, 1, 2, 3).float() \
            if targets.dim() == 4 else \
            F.one_hot(targets, n_classes).permute(0, 3, 1, 2).float()
 
        dims = tuple(range(2, probs.dim()))
        inter = (probs * targets_oh).sum(dims)
        union = probs.sum(dims) + targets_oh.sum(dims)
        dice = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()
 
 
class CombinedLoss(nn.Module):
    """Dice + Cross-Entropy, weighted. Standard for BraTS."""
    def __init__(self, dice_weight=0.5, ce_weight=0.5, class_weights=None):
        super().__init__()
        self.dice = DiceLoss()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dw = dice_weight
        self.cw = ce_weight
 
    def forward(self, logits, targets):
        return self.dw * self.dice(logits, targets) + self.cw * self.ce(logits, targets)

# 4. METRICS

In [8]:
def dice_score(preds: torch.Tensor, targets: torch.Tensor, n_classes: int, smooth: float = 1e-6):
    """
    Returns per-class Dice scores as a list.
    preds  : (B, H, W) long tensor of predicted class indices
    targets: (B, H, W) long tensor of ground-truth class indices
    """
    scores = []
    for c in range(n_classes):
        p = (preds == c).float()
        t = (targets == c).float()
        inter = (p * t).sum()
        scores.append(((2 * inter + smooth) / (p.sum() + t.sum() + smooth)).item())
    return scores

# 5. DATASET (BraTS-style, NIfTI)

In [9]:
try:
    import nibabel as nib
    from pathlib import Path
 
    class BraTSDataset(Dataset):
        """
        Expects folder structure:
            root/
              case_001/
                case_001_t1.nii.gz
                case_001_t1ce.nii.gz
                case_001_t2.nii.gz
                case_001_flair.nii.gz
                case_001_seg.nii.gz   ← labels
              case_002/
                ...
 
        Slices each 3-D volume along the axial axis (dim=2).
        Label mapping:  0=bg, 1=NCR/NET, 2=edema, 4=enhancing → remapped to 3
        """
        MODALITIES = ["t1", "t1ce", "t2", "flair"]
 
        def __init__(self, root: str, slice_axis: int = 2, transform=None):
            self.root = Path(root)
            self.transform = transform
            self.axis = slice_axis
            self.slices = []  # list of (case_dir, slice_idx)
 
            for case in sorted(self.root.iterdir()):
                if not case.is_dir():
                    continue
                seg_path = next(case.glob("*_seg.nii.gz"), None)
                if seg_path is None:
                    continue
                seg = nib.load(str(seg_path)).get_fdata()
                n_slices = seg.shape[slice_axis]
                for s in range(n_slices):
                    self.slices.append((case, s))
 
        def _load_volume(self, path):
            return nib.load(str(path)).get_fdata().astype(np.float32)
 
        def _get_slice(self, vol, idx):
            return np.take(vol, idx, axis=self.axis)
 
        @staticmethod
        def normalize(arr):
            mn, mx = arr.min(), arr.max()
            return (arr - mn) / (mx - mn + 1e-8)
 
        @staticmethod
        def remap_labels(seg):
            """BraTS label 4 → 3 for compact class indexing."""
            out = seg.copy()
            out[seg == 4] = 3
            return out.astype(np.int64)
 
        def __len__(self):
            return len(self.slices)
 
        def __getitem__(self, idx):
            case, sl = self.slices[idx]
            name = case.name
 
            # Stack 4 modalities into (4, H, W)
            mods = []
            for mod in self.MODALITIES:
                vol = self._load_volume(next(case.glob(f"*_{mod}.nii.gz")))
                mods.append(self.normalize(self._get_slice(vol, sl)))
            image = np.stack(mods, axis=0)  # (4, H, W)
 
            # Labels
            seg_vol = self._load_volume(next(case.glob("*_seg.nii.gz")))
            label = self.remap_labels(self._get_slice(seg_vol, sl))
 
            if self.transform:
                # albumentations expects HWC; adapt accordingly
                aug = self.transform(image=image.transpose(1, 2, 0), mask=label)
                image = aug["image"].transpose(2, 0, 1)
                label = aug["mask"]
 
            return torch.tensor(image, dtype=torch.float32), torch.tensor(label, dtype=torch.long)
 
except ImportError:
    BraTSDataset = None  # nibabel not installed; skip dataset class

# 6. TRAINING LOOP

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, deep_supervision=False):
    model.train()
    total_loss = 0.0
 
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
 
        if deep_supervision:
            out_main, out_ds2, out_ds3 = model(images)
            loss = (
                0.6 * criterion(out_main, masks)
                + 0.2 * criterion(out_ds2, masks)
                + 0.2 * criterion(out_ds3, masks)
            )
        else:
            out_main = model(images)
            loss = criterion(out_main, masks)
 
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
 
    return total_loss / len(loader)
 
 
@torch.no_grad()
def validate(model, loader, criterion, device, n_classes):
    model.eval()
    total_loss = 0.0
    all_dice = [0.0] * n_classes
 
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        total_loss += criterion(logits, masks).item()
 
        preds = logits.argmax(dim=1)
        for c, d in enumerate(dice_score(preds, masks, n_classes)):
            all_dice[c] += d
 
    n = len(loader)
    return total_loss / n, [d / n for d in all_dice]
 
 
def train(
    data_root: str,
    epochs: int = 50,
    batch_size: int = 8,
    lr: float = 1e-4,
    n_classes: int = 4,
    device: str = "cuda",
    checkpoint_path: str = "unet_best.pth",
):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")
 
    # ── Model ──
    model = UNet(
        in_channels=4,
        n_classes=n_classes,
        base_filters=64,
        bilinear=True,
        residual=True,
        attention=True,
        deep_supervision=True,
    ).to(device)
 
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {total_params:,}")
 
    # ── Data ──
    if BraTSDataset is None:
        raise ImportError("Install nibabel: pip install nibabel")
 
    train_ds = BraTSDataset(f"{data_root}/train")
    val_ds   = BraTSDataset(f"{data_root}/val")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
 
    # ── Optimizer & scheduler ──
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = CombinedLoss(dice_weight=0.5, ce_weight=0.5)
 
    best_dice = 0.0
    class_names = ["Background", "NCR/NET", "Edema", "Enhancing"]
 
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, deep_supervision=True)
        val_loss, val_dice = validate(model, val_loader, criterion, device, n_classes)
        scheduler.step()
 
        mean_tumor_dice = sum(val_dice[1:]) / (n_classes - 1)  # exclude background
 
        print(f"Epoch {epoch:03d}/{epochs} | "
              f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
              f"Mean tumor Dice: {mean_tumor_dice:.4f}")
        for i, (name, d) in enumerate(zip(class_names, val_dice)):
            print(f"  {name}: {d:.4f}")
 
        if mean_tumor_dice > best_dice:
            best_dice = mean_tumor_dice
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(), "best_dice": best_dice}, checkpoint_path)
            print(f"  ✓ Checkpoint saved (Dice={best_dice:.4f})")
 
    return model

# 7. INFERENCE

In [10]:
@torch.no_grad()
def predict_volume(model, volume: np.ndarray, device="cuda", slice_axis=2):
    """
    Run slice-by-slice inference on a 4-modal MRI volume.
 
    Args:
        model  : trained UNet
        volume : numpy array of shape (4, H, W, D) — 4 modalities
        device : 'cuda' or 'cpu'
        slice_axis: axis along which to slice (default axial = 2)
 
    Returns:
        seg_volume : numpy array of shape (H, W, D) with class indices
    """
    model.eval()
    device = torch.device(device)
    model.to(device)
 
    n_slices = volume.shape[slice_axis + 1]  # +1 for modality dim
    seg = np.zeros(volume.shape[1:], dtype=np.uint8)
 
    for s in range(n_slices):
        slc = np.take(volume, s, axis=slice_axis + 1)  # (4, H, W)
        # Normalize each modality
        slc = np.stack([(m - m.min()) / (m.max() - m.min() + 1e-8) for m in slc])
        t = torch.tensor(slc[None], dtype=torch.float32).to(device)  # (1, 4, H, W)
        logits = model(t)
        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        if slice_axis == 2:
            seg[:, :, s] = pred
        elif slice_axis == 1:
            seg[:, s, :] = pred
        else:
            seg[s, :, :] = pred
 
    # Remap class 3 back to BraTS label 4
    seg[seg == 3] = 4
    return seg

# 8. QUICK SMOKE TEST

In [11]:
if __name__ == "__main__":
    print("=== U-Net Brain Tumor Segmentation — Smoke Test ===\n")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    model = UNet(
        in_channels=4,
        n_classes=4,
        base_filters=32,       # smaller for quick test
        bilinear=True,
        residual=True,
        attention=True,
        deep_supervision=True,
    ).to(device)
 
    x = torch.randn(2, 4, 256, 256).to(device)  # batch=2, 4 modalities, 256×256
    model.train()
    out_main, out_ds2, out_ds3 = model(x)
 
    print(f"Input shape      : {list(x.shape)}")
    print(f"Main output      : {list(out_main.shape)}  (B, n_classes, H, W)")
    print(f"Deep sup output 2: {list(out_ds2.shape)}")
    print(f"Deep sup output 3: {list(out_ds3.shape)}")
 
    # Loss test
    labels = torch.randint(0, 4, (2, 256, 256)).to(device)
    criterion = CombinedLoss()
    loss = (
        0.6 * criterion(out_main, labels)
        + 0.2 * criterion(out_ds2, labels)
        + 0.2 * criterion(out_ds3, labels)
    )
    print(f"\nCombined loss    : {loss.item():.4f}")
 
    # Dice metric test
    preds = out_main.argmax(dim=1)
    scores = dice_score(preds, labels, n_classes=4)
    print(f"Dice per class   : {[f'{s:.4f}' for s in scores]}")
 
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters : {total_params:,}")
    print("\nSmoke test passed ✓")
 
    # To start training:
    # train(data_root="/path/to/brats", epochs=100, batch_size=8, lr=1e-4)

=== U-Net Brain Tumor Segmentation — Smoke Test ===

Input shape      : [2, 4, 256, 256]
Main output      : [2, 4, 256, 256]  (B, n_classes, H, W)
Deep sup output 2: [2, 4, 256, 256]
Deep sup output 3: [2, 4, 256, 256]

Combined loss    : 1.0950
Dice per class   : ['0.1383', '0.3348', '0.2820', '0.1187']

Total parameters : 4,451,524

Smoke test passed ✓
